# AfroXLMR-base — MGT Detection Fine-tuning
**COS760 Group 57 | AfroXLMR-base classifier for isiZulu / English / Combined / Cross-lingual**

---

### Before running:
1. **Enable GPU**: Runtime → Change runtime type → T4 GPU → Save
2. **Run cells top to bottom**, one at a time
3. **To switch experiments**, change `EXPERIMENT` in Cell 4 and re-run from Cell 5 onward

| `EXPERIMENT` | Dataset | Purpose |
|---|---|---|
| `'zul'` | zul_train/val/test.csv | isiZulu-only (RQ1) |
| `'eng'` | eng_train/val/test.csv | English-only baseline |
| `'combined'` | combined_train/val/test.csv | Both languages |
| `'cross'` | Train: eng / Test: zul | Cross-lingual transfer (RQ2) |

## Cell 1 — Install packages

In [ ]:
!pip install transformers datasets evaluate scikit-learn accelerate scipy lime -q
print('Packages installed')

## Cell 2 — Verify GPU
Should print `Tesla T4 | 15.8 GB`. If it shows CPU, go to Runtime → Change runtime type → T4 GPU.

In [ ]:
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {name}  |  {vram:.1f} GB VRAM')
else:
    print('No GPU — go to Runtime → Change runtime type → T4 GPU')

## Cell 3 — Upload CSV files
Select all 9 CSV files at once (hold Ctrl/Cmd to multi-select).

In [ ]:
from google.colab import files
import os

uploaded = files.upload()

os.makedirs('/content/data', exist_ok=True)
for fname in uploaded:
    os.rename(fname, f'/content/data/{fname}')

DATA_DIR = '/content/data'
print('Files uploaded:', os.listdir(DATA_DIR))

## Cell 4 — Experiment configuration
**Change `EXPERIMENT` here to switch between runs.** Re-run from Cell 5 onward after changing it.

In [ ]:
#CHANGE THIS to switch experiments
EXPERIMENT = 'zul'   # 'zul' | 'eng' | 'combined' | 'cross'

MODEL_NAME  = 'Davlan/afro-xlmr-base'
DATA_DIR    = '/content/data'
OUTPUT_DIR  = f'/content/outputs/afroxlmr_{EXPERIMENT}'
BATCH_SIZE  = 16
MAX_LEN     = 256
EPOCHS      = 5
LR          = 2e-5
SEED        = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Experiment : {EXPERIMENT}')
print(f'Model      : {MODEL_NAME}')
print(f'Output     : {OUTPUT_DIR}')

## Cell 5 — Load and inspect data

In [ ]:
import pandas as pd
from datasets import Dataset

def load_csv(name):
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing: {path}')
    return pd.read_csv(path)

if EXPERIMENT == 'cross':
    train_df = load_csv('eng_train.csv')
    val_df   = load_csv('eng_val.csv')
    test_df  = load_csv('zul_test.csv')
else:
    train_df = load_csv(f'{EXPERIMENT}_train.csv')
    val_df   = load_csv(f'{EXPERIMENT}_val.csv')
    test_df  = load_csv(f'{EXPERIMENT}_test.csv')

for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    h = (df['label'] == 0).sum()
    m = (df['label'] == 1).sum()
    print(f'{name:5s}: {len(df)} samples  (human={h}, machine={m})')

## Cell 6 — Tokenise data
Downloads AfroXLMR-base on first run. Subsequent runs use the cached copy.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_hf_dataset(df):
    ds = Dataset.from_dict({
        'text':  df['text'].astype(str).tolist(),
        'label': df['label'].astype(int).tolist(),
    })
    def tok(batch):
        return tokenizer(batch['text'], padding='max_length',
                         truncation=True, max_length=MAX_LEN)
    ds = ds.map(tok, batched=True)
    ds = ds.remove_columns(['text'])
    ds = ds.rename_column('label', 'labels')
    ds.set_format('torch')
    return ds

train_ds = make_hf_dataset(train_df)
val_ds   = make_hf_dataset(val_df)
test_ds  = make_hf_dataset(test_df)
print('Tokenisation complete')

## Cell 7 — Load model

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification

torch.manual_seed(SEED)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True
)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model loaded  ({total_params:.0f}M parameters)')

## Cell 8 — Fine-tune
Trains for up to 5 epochs. Early stopping triggers if validation F1 does not improve for 2 consecutive epochs. The best checkpoint is automatically restored at the end.

In [ ]:
import numpy as np
import scipy.special
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = scipy.special.softmax(logits, axis=-1)[:, 1]
    return {
        'accuracy':  round(accuracy_score(labels, preds), 4),
        'f1':        round(f1_score(labels, preds, average='macro'), 4),
        'precision': round(precision_score(labels, preds, average='macro', zero_division=0), 4),
        'recall':    round(recall_score(labels, preds, average='macro', zero_division=0), 4),
        'roc_auc':   round(roc_auc_score(labels, probs), 4),
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    fp16=False,
    bf16=False,
    seed=SEED,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

best_f1 = trainer.state.best_metric
print(f'\nBest val F1: {best_f1:.4f}', 'Good' if best_f1 > 0.55 else 'Near random — check training logs')

## Cell 9 — Evaluate on test set
Reports accuracy, F1, precision, recall, ROC-AUC, confusion matrix, and majority class baseline.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter

preds_out = trainer.predict(test_ds)
preds  = np.argmax(preds_out.predictions, axis=-1)
labels = preds_out.label_ids
probs  = scipy.special.softmax(preds_out.predictions, axis=-1)[:, 1]

run_name = f'afroxlmr_{EXPERIMENT}'
print('='*55)
print(f'  TEST RESULTS  —  {run_name}')
print('='*55)
print(classification_report(labels, preds, target_names=['Human', 'Machine']))
print(f'ROC-AUC : {roc_auc_score(labels, probs):.4f}')
print('Confusion Matrix:')
print(confusion_matrix(labels, preds))
print('='*55)

majority  = Counter(labels.tolist()).most_common(1)[0][0]
maj_preds = [majority] * len(labels)
print(f'\nMajority baseline F1: {f1_score(labels, maj_preds, average="macro"):.4f}  (your model must beat this)')

## Cell 10 — Save results and download
Saves the test metrics CSV and best model checkpoint. Downloads the CSV to your laptop.

In [ ]:
from google.colab import files as colab_files

results_df = pd.DataFrame([{
    'run':       run_name,
    'model':     'afro-xlmr-base',
    'lang':      EXPERIMENT,
    'accuracy':  accuracy_score(labels, preds),
    'f1_macro':  f1_score(labels, preds, average='macro'),
    'precision': precision_score(labels, preds, average='macro'),
    'recall':    recall_score(labels, preds, average='macro'),
    'roc_auc':   roc_auc_score(labels, probs),
}])

results_path = os.path.join(OUTPUT_DIR, f'test_results_{EXPERIMENT}.csv')
results_df.to_csv(results_path, index=False)
print(results_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

trainer.save_model(os.path.join(OUTPUT_DIR, 'best_model'))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, 'best_model'))

colab_files.download(results_path)

## Cell 11 — LIME explainability
Runs LIME on 10 test samples (5 human, 5 machine) using the model already in memory. Prints the top 12 tokens driving the Machine prediction and downloads a zip of HTML visualisation files.

In [ ]:
from lime.lime_text import LimeTextExplainer
import shutil

LIME_DIR = f'/content/outputs/lime_{run_name}'
os.makedirs(LIME_DIR, exist_ok=True)

def make_predictor(tok, mdl):
    device = next(mdl.parameters()).device
    def predict_proba(texts):
        inputs = tok(
            list(texts), padding=True, truncation=True,
            max_length=MAX_LEN, return_tensors='pt'
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = mdl(**inputs).logits
        return scipy.special.softmax(logits.cpu().numpy(), axis=-1)
    return predict_proba

model.eval()
predictor   = make_predictor(tokenizer, model)
explainer   = LimeTextExplainer(class_names=['Human', 'Machine'])
all_weights = {}
samples     = pd.concat([
    test_df[test_df['label'] == 0].head(5),
    test_df[test_df['label'] == 1].head(5),
])

print(f'Running LIME on {len(samples)} samples — {run_name}')
for i, row in samples.iterrows():
    true_label = 'Human' if row['label'] == 0 else 'Machine'
    exp_result = explainer.explain_instance(
        str(row['text']), predictor,
        num_features=12, num_samples=500, labels=[0, 1]
    )
    for token, weight in exp_result.as_list(label=1):
        all_weights[token] = all_weights.get(token, []) + [weight]
    exp_result.save_to_file(f'{LIME_DIR}/lime_idx{i}_{true_label}.html')

sorted_tokens = sorted(
    {t: np.mean(w) for t, w in all_weights.items()}.items(),
    key=lambda x: abs(x[1]), reverse=True
)[:12]

print(f'\nTop tokens driving Machine prediction — {run_name}')
print(f'{"Token":25s}  {"Avg weight":>10}  Direction')
print('-' * 50)
for token, weight in sorted_tokens:
    print(f'{token:25s}  {weight:+.4f}      {"-> Machine" if weight > 0 else "-> Human"}')

pd.DataFrame([
    {'run': run_name, 'token': t, 'avg_weight': round(np.mean(w), 4),
     'direction': 'Machine' if np.mean(w) > 0 else 'Human'}
    for t, w in all_weights.items()
]).sort_values('avg_weight', key=abs, ascending=False)  .to_csv(f'{LIME_DIR}/lime_tokens.csv', index=False)

shutil.make_archive(f'/content/lime_{run_name}', 'zip', LIME_DIR)
colab_files.download(f'/content/lime_{run_name}.zip')
print('\nOpen any .html file in Chrome to see token highlighting')


## Cell 12 — Qualitative error analysis
Isolates false positives (Human predicted as Machine) and false negatives (Machine predicted as Human), sorted by model confidence. Saves CSVs for manual inspection and report writing.

In [ ]:
ERROR_DIR = f'/content/outputs/errors_{run_name}'
os.makedirs(ERROR_DIR, exist_ok=True)

te_copy = test_df.copy().reset_index(drop=True)
all_probs = scipy.special.softmax(preds_out.predictions, axis=-1)
te_copy['pred_label']   = preds
te_copy['prob_human']   = all_probs[:, 0]
te_copy['prob_machine'] = all_probs[:, 1]
te_copy['correct']      = (te_copy['label'] == te_copy['pred_label'])

fp = te_copy[(te_copy['label'] == 0) & (te_copy['pred_label'] == 1)]     .sort_values('prob_machine', ascending=False).copy()
fn = te_copy[(te_copy['label'] == 1) & (te_copy['pred_label'] == 0)]     .sort_values('prob_human', ascending=False).copy()

total_errors = int((~te_copy['correct']).sum())
print(f'Error analysis — {run_name}')
print(f'  Total errors    : {total_errors} / {len(te_copy)}  ({total_errors/len(te_copy)*100:.1f}%)')
print(f'  False positives : {len(fp)}  (Human predicted as Machine)')
print(f'  False negatives : {len(fn)}  (Machine predicted as Human)')

for label, df_err in [('FALSE POSITIVES — Human predicted as Machine', fp),
                       ('FALSE NEGATIVES — Machine predicted as Human', fn)]:
    conf_col = 'prob_machine' if 'POSITIVE' in label else 'prob_human'
    print(f'\n  {label} (top 5):')
    print(f'  {"Confidence":>10}  Text snippet')
    print(f'  {"-"*65}')
    for _, row in df_err.head(5).iterrows():
        snippet = str(row['text'])[:110].replace('\n', ' ')
        print(f'  {row[conf_col]:>10.4f}  {snippet}...')

fp[['text','label','pred_label','prob_human','prob_machine']]    .to_csv(f'{ERROR_DIR}/false_positives.csv', index=False)
fn[['text','label','pred_label','prob_human','prob_machine']]    .to_csv(f'{ERROR_DIR}/false_negatives.csv', index=False)
pd.DataFrame([{
    'run': run_name, 'total_errors': total_errors,
    'false_positives': len(fp), 'false_negatives': len(fn),
    'error_rate': round(total_errors / len(te_copy), 4),
}]).to_csv(f'{ERROR_DIR}/error_summary.csv', index=False)

print(f'\nSaved to {ERROR_DIR}')
print('  Open false_positives.csv and false_negatives.csv to select examples for your report.')


---
## Cell 13 — (Optional) Run all 4 experiments sequentially
**If the session disconnects you will lose all progress. Running Cells 4–10 one experiment at a time is safer.**

Requires Cells 1–6 to have been run first.

In [ ]:
# !! Only run this if you want to do all 4 experiments automatically !!

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
)
from collections import Counter
from lime.lime_text import LimeTextExplainer
import shutil

all_results    = []
all_lime_rows  = []
all_error_rows = []

def make_predictor(tok, mdl):
    device = next(mdl.parameters()).device
    def predict_proba(texts):
        inputs = tok(
            list(texts), padding=True, truncation=True,
            max_length=MAX_LEN, return_tensors='pt'
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = mdl(**inputs).logits
        return scipy.special.softmax(logits.cpu().numpy(), axis=-1)
    return predict_proba

for exp in ['zul', 'eng', 'combined', 'cross']:
    print(f'\n{"="*55}\n  Experiment: {exp}\n{"="*55}')

    if exp == 'cross':
        tr = load_csv('eng_train.csv')
        va = load_csv('eng_val.csv')
        te = load_csv('zul_test.csv')
    else:
        tr = load_csv(f'{exp}_train.csv')
        va = load_csv(f'{exp}_val.csv')
        te = load_csv(f'{exp}_test.csv')

    tr_ds = make_hf_dataset(tr)
    va_ds = make_hf_dataset(va)
    te_ds = make_hf_dataset(te)

    model_run = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True
    )

    out = f'/content/outputs/afroxlmr_{exp}'
    os.makedirs(out, exist_ok=True)

    args_run = TrainingArguments(
        output_dir=out,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=LR,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        greater_is_better=True,
        logging_steps=50,
        fp16=False,
        bf16=False,
        seed=SEED,
        report_to='none',
    )

    t = Trainer(
        model=model_run, args=args_run,
        train_dataset=tr_ds, eval_dataset=va_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    t.train()

    best_f1 = t.state.best_metric
    print(f'  Best val F1: {best_f1:.4f}', 'Good' if best_f1 > 0.55 else 'Near random')

    p_out  = t.predict(te_ds)
    preds  = np.argmax(p_out.predictions, axis=-1)
    labels = p_out.label_ids
    probs  = scipy.special.softmax(p_out.predictions, axis=-1)[:, 1]

    majority  = Counter(labels.tolist()).most_common(1)[0][0]
    maj_preds = [majority] * len(labels)

    result = {
        'run':         f'afroxlmr_{exp}',
        'model':       'afro-xlmr-base',
        'lang':        exp,
        'accuracy':    accuracy_score(labels, preds),
        'f1_macro':    f1_score(labels, preds, average='macro'),
        'precision':   precision_score(labels, preds, average='macro'),
        'recall':      recall_score(labels, preds, average='macro'),
        'roc_auc':     roc_auc_score(labels, probs),
        'majority_f1': f1_score(labels, maj_preds, average='macro'),
    }
    all_results.append(result)

    pd.DataFrame([result]).to_csv(f'{out}/test_results_{exp}.csv', index=False)
    t.save_model(f'{out}/best_model')
    tokenizer.save_pretrained(f'{out}/best_model')
    for ckpt in __import__('glob').glob(f'{out}/checkpoint-*'):
        shutil.rmtree(ckpt, ignore_errors=True)

    print(f'  Test F1: {result["f1_macro"]:.4f}  |  Majority baseline: {result["majority_f1"]:.4f}')

    # LIME
    print(f'\n  Running LIME...')
    lime_dir = f'/content/outputs/lime_afroxlmr_{exp}'
    os.makedirs(lime_dir, exist_ok=True)

    from transformers import AutoTokenizer as AT, AutoModelForSequenceClassification as AM
    lime_tok = AT.from_pretrained(f'{out}/best_model')
    lime_mdl = AM.from_pretrained(f'{out}/best_model')
    lime_mdl.eval()

    predictor   = make_predictor(lime_tok, lime_mdl)
    explainer   = LimeTextExplainer(class_names=['Human', 'Machine'])
    all_weights = {}
    samples     = pd.concat([te[te['label'] == 0].head(5), te[te['label'] == 1].head(5)])

    for i, row in samples.iterrows():
        true_label = 'Human' if row['label'] == 0 else 'Machine'
        exp_result = explainer.explain_instance(
            str(row['text']), predictor,
            num_features=12, num_samples=500, labels=[0, 1]
        )
        for token, weight in exp_result.as_list(label=1):
            all_weights[token] = all_weights.get(token, []) + [weight]
        exp_result.save_to_file(f'{lime_dir}/lime_idx{i}_{true_label}.html')

    sorted_tokens = sorted(
        {tk: np.mean(w) for tk, w in all_weights.items()}.items(),
        key=lambda x: abs(x[1]), reverse=True
    )[:12]

    print(f'  Top tokens — {exp}')
    print(f'  {"Token":25s}  {"Avg weight":>10}  Direction')
    print(f'  {"-"*50}')
    for token, weight in sorted_tokens:
        direction = 'Machine' if weight > 0 else 'Human'
        print(f'  {token:25s}  {weight:+.4f}      -> {direction}')
        all_lime_rows.append({
            'experiment': exp, 'model': 'afro-xlmr-base',
            'token': token, 'avg_weight': round(weight, 4), 'direction': direction,
        })

    pd.DataFrame([
        {'experiment': exp, 'model': 'afro-xlmr-base', 'token': t,
          'avg_weight': round(np.mean(w), 4),
          'direction': 'Machine' if np.mean(w) > 0 else 'Human'}
        for t, w in all_weights.items()
    ]).sort_values('avg_weight', key=abs, ascending=False)      .to_csv(f'{lime_dir}/lime_tokens.csv', index=False)

    del lime_mdl, lime_tok

    # Error Analysis
    print(f'\n  Running error analysis...')
    error_dir = f'/content/outputs/errors_afroxlmr_{exp}'
    os.makedirs(error_dir, exist_ok=True)

    te_copy = te.copy().reset_index(drop=True)
    all_probs_arr = scipy.special.softmax(p_out.predictions, axis=-1)
    te_copy['pred_label']   = preds
    te_copy['prob_human']   = all_probs_arr[:, 0]
    te_copy['prob_machine'] = all_probs_arr[:, 1]
    te_copy['correct']      = (te_copy['label'] == te_copy['pred_label'])

    fp = te_copy[(te_copy['label'] == 0) & (te_copy['pred_label'] == 1)]         .sort_values('prob_machine', ascending=False)
    fn = te_copy[(te_copy['label'] == 1) & (te_copy['pred_label'] == 0)]         .sort_values('prob_human', ascending=False)

    total_errors = int((~te_copy['correct']).sum())
    print(f'  Errors: {total_errors} total  |  FP={len(fp)}  FN={len(fn)}')

    fp[['text','label','pred_label','prob_human','prob_machine']]        .to_csv(f'{error_dir}/false_positives.csv', index=False)
    fn[['text','label','pred_label','prob_human','prob_machine']]        .to_csv(f'{error_dir}/false_negatives.csv', index=False)
    pd.DataFrame([{
        'run': f'afroxlmr_{exp}', 'model': 'afro-xlmr-base',
        'total_errors': total_errors, 'false_positives': len(fp),
        'false_negatives': len(fn),
        'error_rate': round(total_errors / len(te_copy), 4),
    }]).to_csv(f'{error_dir}/error_summary.csv', index=False)

    all_error_rows.append({
        'run': f'afroxlmr_{exp}', 'model': 'afro-xlmr-base', 'lang': exp,
        'total_errors': total_errors, 'false_positives': len(fp),
        'false_negatives': len(fn),
        'error_rate': round(total_errors / len(te_copy), 4),
    })

    del model_run, t
    torch.cuda.empty_cache()

# Save all outputs 
os.makedirs('/content/outputs', exist_ok=True)

summary = pd.DataFrame(all_results)
print('\n' + '='*70)
print(summary[['run','f1_macro','roc_auc','accuracy','majority_f1']].to_string(
    index=False, float_format=lambda x: f'{x:.4f}'
))

summary.to_csv('/content/outputs/all_afroxlmr_results.csv', index=False)
pd.DataFrame(all_lime_rows).to_csv('/content/outputs/all_afroxlmr_lime_tokens.csv', index=False)
pd.DataFrame(all_error_rows).to_csv('/content/outputs/all_afroxlmr_error_summary.csv', index=False)

# Zip only the LIME HTML directories
lime_dirs = __import__('glob').glob('/content/outputs/lime_afroxlmr_*')
with __import__('zipfile').ZipFile('/content/outputs/all_afroxlmr_lime.zip', 'w',
                                   __import__('zipfile').ZIP_DEFLATED) as zf:
    for lime_d in lime_dirs:
        for html_f in __import__('glob').glob(f'{lime_d}/*.html'):
            zf.write(html_f, __import__('os').path.relpath(html_f, '/content/outputs'))

colab_files.download('/content/outputs/all_afroxlmr_results.csv')
colab_files.download('/content/outputs/all_afroxlmr_lime_tokens.csv')
colab_files.download('/content/outputs/all_afroxlmr_error_summary.csv')
colab_files.download('/content/outputs/all_afroxlmr_lime.zip')
print('\nAll results, LIME tokens, error summaries, and HTML files downloaded.')
